In [1]:
from dataclasses import dataclass
from torch import nn
import torch

In [2]:
@dataclass
class ModelParameters:
    vocab_size: int  # total token in vocabulary
    d_model:int   # embedding dimension
    max_seq_len:int  # context window 
    num_layers: int # here layers mean decoder block
    num_heads: int # no. of attention head

In [3]:
class EmbeddingLayer(nn.Module):
    def __init__(self,model_params:ModelParameters):
        super().__init__()

        """
        Always remember that embedding layer is not actually a neural laer with any activation function ,
        its just an 2d matrix lookup the matrix size is (vocab_size,d_model) but but but the d_model here is lets one array
         so that array is already have learnable parameter means in our case that 32 dim arrays have the values from the training 
         for e.g:
         token_embeding[1] will be the embeding of token id 1 which is 32 dim embedding array
        """
        self.token_embeddings=nn.Embedding(model_params.vocab_size,model_params.d_model)
        self.pos_embeddings = nn.Embedding(model_params.max_seq_len,model_params.d_model)

    def forward(self,token_ids):
        pos_ids = torch.arange(token_ids.shape[1])
        embedding = self.token_embeddings(token_ids) + self.pos_embeddings(pos_ids)
        return embedding


In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self,model_params:ModelParameters):
        super().__init__()

        self.config = model_params
        # initialize q,k,v
        self.w_q = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_k = nn.Linear(self.config.d_model,self.config.d_model,bias=False)
        self.w_v = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

        self.register_buffer(
                            "casual_mask", 
                            torch.tril(
                                torch.ones(
                                    self.config.max_seq_len, 
                                    self.config.max_seq_len
                                    )
                                )
                        )
        
        self.head_dim = self.config.d_model//self.config.num_heads
        

        self.w_o = nn.Linear(self.config.d_model,self.config.d_model,bias=False)

    def forward(self,x):
        # here x is the embeding and not raw x  and b is batch size, t is the current sequence length and d_model is hidden dimension
        # shape of x -> (b,t,d_model)
        q = self.w_q(x) # (b,t,d_model)
        k = self.w_k(x) # (b,t,d_model)
        v = self.w_v(x) # (b,t,d_model)

        q = torch.reshape(q,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        k = torch.reshape(k,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)
        v = torch.reshape(v,(x.shape[0],x.shape[1],self.config.num_heads,self.head_dim))  # (b,t,num_head,head_dim)


        q = q.transpose(1,2) # (b,num_head,t,head_dim)
        k = k.transpose(1,2) # (b,num_head,t,head_dim)
        v = v.transpose(1,2) # (b,num_head,t,head_dim)

        scores = q @ k.transpose(-2,-1) / (self.head_dim ** 0.5) # (b,num_head,t,t)
        mask = self.casual_mask[:x.shape[1],:x.shape[1]] # (t,t)
        masked_score = scores.masked_fill(mask==0,float("-inf")) # (b,num_head,t,t)
        attention_weights = torch.softmax(masked_score,dim = -1) # (b,num_head,t,t)
        context_vector = attention_weights @ v # (b,num_head,t,head_dim)

        context_vector = context_vector.transpose(1,2) # (b,t,num_head,head_dim) 
        context_vector = context_vector.contiguous().view(x.shape) # (b,t,d_model)
        return self.w_o(context_vector) #(b,t,d_model) which is same as the input embedding 


In [5]:
model_params = ModelParameters(
    vocab_size=44,
    d_model = 32,
    max_seq_len=5,
    num_layers=1,
    num_heads=4
)

In [6]:
embedding_layer = EmbeddingLayer(model_params)

In [7]:
x = torch.randint(0,model_params.vocab_size,(2,5))
x_embed = embedding_layer(x)
x_embed.shape

torch.Size([2, 5, 32])

In [13]:
multi_head_attention_layer  = MultiHeadAttention(model_params)
output = multi_head_attention_layer(x_embed)
output.shape

torch.Size([2, 5, 32])

In [14]:
class FeedForward(nn.Module):
    def __init__(self,model_params:ModelParameters):
        super().__init__()
        self.config = model_params
        self.model = nn.Sequential(
            nn.Linear(self.config.d_model,4*self.config.d_model),
            nn.GELU(),
            nn.Linear(4*self.config.d_model,self.config.d_model)
        )

    def forward(self,x):
        return self.model(x)

In [15]:
ffn = FeedForward(model_params)

In [16]:
output = ffn(output)

In [17]:
output.shape

torch.Size([2, 5, 32])

In [18]:
norm_layer = nn.LayerNorm(model_params.d_model)
output_norm = norm_layer(output)

In [19]:
class DecoderBlock(nn.Module):
    def __init__(self,model_params:ModelParameters):
        super().__init__()

        self.config = model_params
        self.attention_layer = MultiHeadAttention(model_params)
        self.ffn = FeedForward(model_params)
        self.ln_1 =  nn.LayerNorm(model_params.d_model) # before attention layer
        self.ln_2 =  nn.LayerNorm(model_params.d_model) # before feed forward layer

    def forward(self,x):
        x =  x+self.attention_layer(self.ln_1(x))
        x = x+ self.ffn(self.ln_2(x))
        return x



In [20]:
decoder_block = DecoderBlock(model_params)
out = decoder_block(x_embed)
out.shape

torch.Size([2, 5, 32])

In [22]:
total_params = sum(p.numel() for p in decoder_block.parameters())
trainable_params = sum(p.numel() for p in decoder_block.parameters() if p.requires_grad)
print(f"Total Parameters in 1 Decoder Block: {total_params:,}")

Total Parameters in 1 Decoder Block: 12,576


In [24]:
print(decoder_block)

DecoderBlock(
  (attention_layer): MultiHeadAttention(
    (w_q): Linear(in_features=32, out_features=32, bias=False)
    (w_k): Linear(in_features=32, out_features=32, bias=False)
    (w_v): Linear(in_features=32, out_features=32, bias=False)
    (w_o): Linear(in_features=32, out_features=32, bias=False)
  )
  (ffn): FeedForward(
    (model): Sequential(
      (0): Linear(in_features=32, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Linear(in_features=128, out_features=32, bias=True)
    )
  )
  (ln_1): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
  (ln_2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
)


```mermaid
flowchart TD
    In["Input: x (B, T, 32)"] --> LN1["LayerNorm 1"]
    In --> Add1["(+) Residual 1"]
    LN1 --> MHA["MultiHeadAttention (4 heads)"]
    MHA --> Add1
    Add1 --> LN2["LayerNorm 2"]
    Add1 --> Add2["(+) Residual 2"]
    LN2 --> FFN["FeedForward (32 -> 128 -> 32)"]
    FFN --> Add2
    Add2 --> Out["Output (B, T, 32)"]